# ROI Averaging

This notebook applies ROI Averaging to the preprocessed data (epoched data only). It can average over electrodes and time windows. 

- **input:** folder with FIF files that contain epoched data  
- **output:**  
    1. csv file with averaged channels for all timepoints in the epoch, 
    2. png with a plot of averaged channels, 
    3. csv file with averaged values for channels and time windows.

### Getting started

- Modify the values in the cell marked with `# --------- ADJUST VALUES BELOW AS NEEDED ---------`.
- Make sure `roi_groups` is a dictionary with a label as key and a list of electrodes as value. Just like the example.
- Set `write_full_timecourse` to `false` if you don't want the averaged channels for all the timepoints in the epochs (output 1 and 2). 
- Leave `window_analyses` empty if you don't wish to include averaging over time windows (output 2). Else, make sure it has a `name`, `roi`, `tmin` and `tmax`. Make sure `roi` matches the label (key) you've set in `roi_groups`. Just like the example.



In [1]:
# --------- ADJUST VALUES BELOW AS NEEDED ---------

# folder containing input files for ROI Averaging (*.fif)
input_dir = './output_dir'

# folder where ROI-averaged files are written
output_dir = './output_ROI_Average'

## Define region(s) of interest (ROI): 
# electrode group. name -> list of electrode labels.

roi_groups = {
    'posterior': [
        'C1', 'C3', 'C5', 'T7', 'CP1', 'CP3', 'CP5',
        'TP7', 'P1', 'P3', 'P5', 'P7', 'PO3', 'PO7',
        'O1', 'C2', 'C4', 'C6', 'T8', 'CP2', 'CP4',
        'CP6', 'TP8', 'P2', 'P4', 'P6', 'P8', 'PO4',
        'PO8', 'O2', 'Cz', 'CPz', 'Pz', 'POz', 'Oz'],
    'anterior': [
        'Fp1', 'AF3', 'AF7', 'F1', 'F3', 'F5', 'F7',
        'FC1', 'FC3', 'FC5', 'FT7', 'Fp2', 'AF4',
        'AF8', 'F2', 'F4', 'F6', 'F8', 'FC2', 'FC4',
        'FC6', 'FT8', 'AFz', 'Fz'],
    # 'FPz', 'FCz'
}

# OUTPUT 1: write the full per-timepoint time-course to CSV? Set to False to skip.
write_full_timecourse = True

# OUTPUT 2: time-window analyses. Each entry averages one ROI group's
# signal over a time window (seconds) into a single value per epoch.
# Combine any ROI defined above with any time window you need.
# Make sure the ROI label exists in the roi_groups.
# Leave the list empty ([]) to skip this output.
window_analyses = [
    {'name': 'P2_anterior_200_250', 'roi': 'anterior', 'tmin': 0.200, 'tmax': 0.250},
    {'name': 'N400_posterior_350_450', 'roi': 'posterior', 'tmin': 0.350, 'tmax': 0.450},
    {'name': 'LPP_posterior_500_650', 'roi': 'posterior', 'tmin': 0.500, 'tmax': 0.650},
]


In [2]:
from pathlib import Path
import mne
import pandas as pd

INPUT_DIR = Path(input_dir)
OUTPUT_DIR = Path(output_dir)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
fif_files = sorted(INPUT_DIR.glob('*.fif'))
if not fif_files:
    raise FileNotFoundError(f'No .fif files found in {INPUT_DIR.resolve()}')

epoched_files = []
print(f'Found {len(fif_files)} FIF file(s):')
for i, f in enumerate(fif_files):
    print(f'[{i}] {f.name} -> {mne.what(f)}')

print(f'\nIt will only use epoched files for ROI Averaging.')
for i, f in enumerate(fif_files):
    if mne.what(f) == 'epochs':
        print(f'[{i}] {f.name}')
        epoched_files.append(f)

Found 2 FIF file(s):
[0] VBTS_processed_data-part1.fif -> raw
[1] WJMT_processed_data.fif -> epochs

It will only use epoched files for ROI Averaging.
[1] WJMT_processed_data.fif


In [4]:
import matplotlib
mne.viz.set_browser_backend('matplotlib')

def get_condition_series(epochs):
    """
    Map each epoch's event code to its condition label.
    If the label doesn't exist it will just show the code. 
    """
    id_to_label = {code: label for label, code in epochs.event_id.items()}
    return [id_to_label[code] for code in epochs.events[:, -1]]

def process_file(fif_path, roi_groups, write_full_timecourse, window_analyses, output_dir):
    epochs = mne.read_epochs(fif_path, preload=True)

    roi_picks = {
        name: mne.pick_channels(epochs.ch_names, include=channels)
        for name, channels in roi_groups.items()
    }
    epochs_roi = mne.channels.combine_channels(epochs, groups=roi_picks, method='mean')

    timecourse_df, window_df = None, None
    stem = fif_path.stem

    if write_full_timecourse:
        timecourse_df = epochs_roi.to_data_frame()
        timecourse_df.to_csv(output_dir / f'{stem}_roi_timecourse.csv', index=False)

        # create and save the plot
        matplotlib.use('Agg')
        fig = epochs_roi.plot(n_epochs=len(epochs)) 
        if len(epochs) > 50:
            fig.set_figwidth(len(epochs)*2)
        fig.savefig(output_dir / f'{stem}_roi_plot.png')

    # Average over time window (averaging over channels in done above)
    if window_analyses:
        conditions = get_condition_series(epochs)
        window_df = pd.DataFrame({ 'condition': conditions, 'epoch': range(len(epochs))})
        if epochs.metadata is not None:
            window_df = pd.concat([window_df, epochs.metadata.reset_index(drop=True)], axis=1)

        for analysis in window_analyses:
            roi_name = analysis['roi']
            if roi_name not in roi_groups:
                raise KeyError(f"Unknown ROI '{roi_name}' in window_analyses; must be one of {list(roi_groups)}")
            window_data = epochs_roi.copy().crop(tmin=analysis['tmin'], tmax=analysis['tmax']).get_data(picks=[roi_name])
            window_df[analysis['name']] = window_data.mean(axis=(1, 2))

        window_df.to_csv(output_dir / f'{stem}_roi_window_averages.csv', index=False)

    return timecourse_df, window_df


Using matplotlib as 2D backend.


In [ ]:
results = {}
for fif_path in epoched_files:
    print(f'Processing {fif_path.name}...')
    results[fif_path.stem] = process_file(
        fif_path, roi_groups, write_full_timecourse, window_analyses, OUTPUT_DIR
    )

print(f'Done. Output files written to {OUTPUT_DIR.resolve()}')


In [6]:
# Preview the full time-course output (OUTPUT 1) for the first processed file
preview_stem = epoched_files[0].stem if epoched_files else None
if preview_stem and results[preview_stem][0] is not None:
    display(results[preview_stem][0].head())

,time,condition,epoch,posterior,anterior
0,-0.200,een deelneemster 1pN,0,4.940550,6.271353
1,-0.198,een deelneemster 1pN,0,4.496525,5.529146
2,-0.196,een deelneemster 1pN,0,3.918569,4.621769
3,-0.194,een deelneemster 1pN,0,3.190790,3.592979
4,-0.192,een deelneemster 1pN,0,2.324852,2.486648


In [7]:
# Preview the per-epoch window-average output (OUTPUT 2) for the first processed file
if preview_stem and results[preview_stem][1] is not None:
    display(results[preview_stem][1].head())


,condition,epoch,P2_anterior_200_250,N400_posterior_350_450,LPP_posterior_500_650
0,een deelneemster 1pN,0,5.991553e-06,0.000012,0.000009
1,start CW-1,1,5.592309e-06,-0.000008,-0.000017
2,start CW 1pN,2,-1.213897e-05,0.000015,0.000011
3,een vrouw 1pN,3,1.837699e-05,-0.000003,0.000007
4,start CW-1,4,1.131630e-07,-0.000021,-0.000026
